# RGBMARGIN1 — Auliya fold A, seed 0
Satu YOLO26s-seg native dengan tambahan loss terhadap pesaing kelas terkuat pada anchor positif. Tidak ada parameter, fitur, atau komputasi inferensi tambahan. Input tetap RGB 640 dan mask tetap stride 4. Baseline RGB lama dipakai untuk evaluasi, bukan sebagai bobot awal: kandidat mulai dari bobot YOLO resmi.

Notebook memakai ZIP Auliya dan hasil RGB yang sudah ada di Drive bersama. Jalankan preflight terlebih dahulu. Training memerlukan opt-in dan runtime yang cocok; test tidak dibuka.

In [ ]:
from google.colab import drive
from pathlib import Path, PurePosixPath
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib, json, os, shutil, subprocess, sys, time

ARM = 'RGBMARGIN1'
drive.mount('/content/drive')
ROOT = Path('/content/drive')
PILOT_NAME = 'auliya-robusta-sni-frequency-colab-v1.zip'
BASE_NAME = 'auliya-official-fold-a-haar-mask2-source-v1.zip'
RGB_REL = Path('experiments/auliya-official-fold-a-frequency-v1/RGB')
canonical = ROOT/'MyDrive/Coffee_Bean_Detection'
shared = sorted((ROOT/'.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection'))
required = (Path('bundles')/PILOT_NAME, Path('bundles')/BASE_NAME,
            RGB_REL/'validation_result.json', RGB_REL/'weights/best.pt')
matches = {p.resolve(): p for p in shared if all((p/f).is_file() for f in required)}
if all((canonical/f).is_file() for f in required):
    PROJECT = canonical
elif len(matches) == 1:
    PROJECT = next(iter(matches.values()))
else:
    raise RuntimeError(f'Butuh folder proyek Auliya dengan ZIP dan RGB reference. Ditemukan: {list(matches.values())}')
BASE = PROJECT/'bundles'/BASE_NAME
PILOT = PROJECT/'bundles'/PILOT_NAME
RGB = PROJECT/RGB_REL
OUT = PROJECT/'experiments/auliya-official-fold-a-rgbmargin1-v1'

def digest(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def fetch(url, target, expected):
    if not target.exists():
        part = target.with_suffix(target.suffix + '.part')
        with urlopen(url, timeout=60) as remote, part.open('wb') as local:
            shutil.copyfileobj(remote, local, length=4*1024*1024)
        if digest(part) != expected:
            part.unlink()
            raise ValueError(f'SHA256 unduhan tidak cocok: {target.name}')
        part.replace(target)
    if digest(target) != expected:
        raise ValueError(f'SHA256 tidak cocok: {target.name}')
    return target

def extract_sources(archive, target):
    with ZipFile(archive) as source:
        for name in source.namelist():
            p = PurePosixPath(name)
            if p.parts[:2] != ('src', 'coffee_detector') or '..' in p.parts or p.suffix != '.py':
                raise ValueError(f'Entri paket kode tidak aman: {name}')
        source.extractall(target)

BASE_SHA = '6c32acac9c38c7bcbbd42b33ddb93f4035e45b6ecbc7bb4bf51f4d72cef17b79'
OVERLAY_SHA = '6071024e076cd096c25e3de4d1f961296cf347b1746546166854dd24b63a0831'
if digest(BASE) != BASE_SHA:
    raise ValueError('Paket kode Auliya dasar tidak cocok dengan protokol.')
WORK = Path('/content')/f'auliya-rgbmargin1-{OVERLAY_SHA[:12]}'
WORK.mkdir(exist_ok=True)
print('PROJECT:', PROJECT, '| RGB:', RGB, '| CANDIDATE:', ARM)

In [ ]:
# Overlay kode kandidat, agar notebook tidak perlu clone GitHub atau upload paket baru.
import base64, io

OVERLAY_B64 = """
UEsDBBQAAAAIAAAAe13E7PGntgQAAIQKAAAqAAAAc3JjL2NvZmZlZV9kZXRlY3Rvci9hdWxpeWFfcml2YWxfbWFyZ2luLnB5pVZda+s2GL73rxDa
xewe22syOBc5ZNCV0hXaZuSEjRGCUGzZFrWlTJLT5hzOf9+rj9hJW3ozQ4gt6Xne71cvxnilKBeZFO0BaaOkqJk2meJ72qKGwxeSAlGteS1Yif5Z
3C+mnzPNarSTmhu+ZzrHGEdRpWSHCKl60ytGCOLdTiqDqBDSUMOl0FEU1jpqmuHDSFU0Ht23RtH2YHihcyHyTpZ9C/QNo+WR7iurOybM9PO7CEP1
k3511Ml+kCVr30J6w1udt1IPoJvpzT18pke0/QBh0fLur6t7crV8QHOEl7e/P1wtb+8eJzhs/LlcrBbXi3u7S/uWH2gmq4oXnLZZJdsyo5mqtx1V
NReTbD/g/r65u/1jBah8chmWPDMsWS+BbnU8zZOwd714XC2vri3ge4TgwU9clHiG8DEYhIqikYoMkSTP9oUULYSQABs3xEUVp55AsZY6oJFE+Lei
1aSGnADeUy0DwHN4U4YTXulwYqusEkzD7hpLwaYdFQecIvcOP7xJox9RFJWsGpKIuIQLtLEupGJ65nMjXzGhpUpRBYu1kr0oX+04se8/BgiZIbUh
4Cj28ppyS03RzFDJC5Og7Lez3Zm3BuMHRsVYCkPaf0Femcyo3jQI+HnBNOo1Q96RaMdUxjsKuB0tSyBwlWJZf3KF9LNGO8V2SgJOx4mFavTMDcgK
OEgizQxkp0CmYUjTDtZUyVTuaNwhbxmkhDNmjd0f4eUL3uTgxYbuWJxNktzIuGR70HHu3Zv7rxSV5rBjc296C8mSOG5wmUsaUGmghsz4f6QFOMxY
Qr+x5cKtxCeGpKjjomWiNs2R04lbX248x9En88CWF32n+y6+hAiGJXeupVvW2mOjJev4VT441cDzn46s61mKHiFHN8l6TLfNxhPazLeEQS3oI0JD
OrB4kqLpGSCvWklN7BWG/GCA8vC8hrJmykK8gqPEXP/bM/YN6DzOlYQekbqgxrwLhdpw8nDGRYWTJKcdfYlL3s2PVAy6skDxWYv5dJSQORUTCGzb
x6AGODNBv5zUm1/KwYvdjkB4rIZR5JyKlpbjwdXtSc+Mz/pn4mvJFrz1/vE2IT4aGppWSWwXjjVrq9QWRalDcQaofTzMsoIDXJPmhnUuID2UGij4
MfkZ7cA6Wpm+7hbAPMpcz6abAeT7FOy/37+cpDX2eYI36UdC3ihUhJhrvZ5sIEjWJ3lz2OV24+KsJcOnlziAQ6BPXeVLTRtaPMWxo70EjYDMe3E9
3SRJir5fXDhvQpu2/d/uQP+GV6hogDZx8uODkI+XbPxmJRka6VfbvsAVz1SVMBiUMG9QA10NLouOgs6ux8HlCh4Bo7lgX5B+pjvkZhO7VyjQUQGx
HzmOOUXAk3AnkZA+RVWn6AL+m/mvKRLF3NZIivZMbaFY5yub62NWHXNnIHFwwBaNA4tihIb/MVah0c99kDpr7jqbjHnCK2T7YOzPJYhrBOPQOMVA
M7cVAXWtwpnxmiTFfop9gTugfZmdXXUwuMFlswIBN0pJFeOblx0rDIxp1llBt7KHGcRfyWF6A2XHECE7XeHRImdIpRi0IlEcCFUdWDdMP6PPnbOG
eDjPnzg1pGGYpkJc3u8USfQfUEsDBBQAAAAIAAAAe12q8J764ggAAHAWAAA8AAAAc3JjL2NvZmZlZV9kZXRlY3Rvci9leHBlcmltZW50cy9hdWRp
dF9hdWxpeWFfcml2YWxfbWFyZ2luLnB5pVhtb+M2Ev7uX0GwX+SDoia5vWKbgwvsZbe9AulmkWQLHAyDoCXaViOLOpJK4ga5395nSEpW5GTbxS2Q
tSgO532eGYpz/qsy5WrH3EaxWrryTjGr1ltVOyx0zdZGNhsm64KVVlfSqUCp7lmlrWVLtdIG74ws67JeZ5zzyWRl9JYJsWpda5QQrNw22jgwqXXg
aieT+O43q+tAn+tm11EWSjW0DjuNdJuqXHabn7AMG05tm1VZqW7nRtGvNLv3pVG50yYycLtG2Y7oGr+V+ii3yjYyV70mIM8j37aCPdXOlXl/6j+X
F5cHm1m+WncEa+UElpPOmNVKKVEo5/XIZFuVOylMeScrsZVmXdbdwauff313Id5d/ZLGx/PLjzdX785vuvWnq8uby/PLC6zp+C/+9HWI0QVi8LJE
9dAgsERjM9PWImoAqjIvocTKqP+2qs57nycThn/X5//+8P7zxYeUCVVbCh8OO/BJQ4yVEQh4yqBIWSAbBgx1VaST6WQyKdSKiW1rXaJvU2aURIzP
mHVmyo5+YB91rc68rHLFkBBM34Yl/YMIq9hVEPnBGG2ScL7j2xi1qsr1xj3zZVKoO1Xphsw98wmSsntFZLZb/i1FVt2VufKasBnjedNyr1FR5i6o
MLI5iTymfjMcx0mfKVlYJiuet4U8ewzLJ05WheestIXKy62skilTFewK76ed8ZGMspPNSCEw4r7UyC1BCr0DIyHvZFnJZaWS6Re9xc8/v3/H2rqn
/ydrQXN0FLWH0QwBZOefPu99yaOB0kmY93Johy7OjAIW3EGXeNDiGFVyVmlZ2CQhTlkjDWjZt4w/YyRyXSOVcpfRCT4FM1kIpx5cgnTUBVBkxlu3
OnrLI3urW+MdT0WYIHp9XJCU0t7OeEQsMNvqQlX+lH8ixeYLvyarpdkyVF7Cr376F0/3pTfwacxynNPL31BKmRAAOyGSQfon4BN1GxzJov8eeZ3z
M3Z6kiKkG4Ae1MD671jWhDp4poyDte1WGTg6Kew8bi2m06cDvkhxsmMEXEleWWHUVjazG9OqvTr3pdvE7DFIJr3NoPKtMPU6CUlgZ/PFwGIvakiO
OgNaOLFWNekHMNnKukX4rFJF0iHEnNMSGj9j5L1ONRJ1J1j075IQxWwnt1UaQwoYUWaprZr9KFEfe07+RGd3RNZEg9iUBdR/5BR2+LEPPHyrGp1v
yLl7BeOrBXa3OCAMNR9QvHkaibKZbBpVF0kQ7HT01DRTd1S+gTy0x5TlcJOvECgXjoc0lbBriXeBLrOOisiHejo49Ox9QB2PlVZmt2pngRXAArvs
FgQHsqqSECFgNvTxgua3i6lPamCspKwGgxIdEYem6SAm/LxXl+R4bJFFwagcKodMl/Wu7/zON3igoDYRFB56wKPsSJLTlDL5++9SdnL6Fnb1STIL
VD9162T6LG1Ovp9OB46djDK1rFcKcJErny1DkNvBWiN7tyYP+9jtltgij/fOHe6Sb5ZQOt+EmkffOYVGO0oWesYfH5VBiMTY3UbOA58FfE4SB8sY
giDB5piFLPFf6ofw4PMu18ojYMzUxmin+fMo9UrzG+Cho6RmBB5rVXTBMfKePQbJT0y3rmmd5XtzX9YdzlsGHemJ1Py9bBLyKTkPWIo0DPiBWB7I
tnTyXppiJM8H7nDaSqijlA8zbtbL0JhPjvomc4RmK62f2fZOp9EO8aMWndDOlNqFUdRGssbxPRR6k6xE/B+5rzlC0TglJn34pxn6G1Ub9/gjCEFA
eIgXT0+pl713H4mErwdNxu+/0mEOIWLEqAOvfdn7F+ybvsEgUxylCyudxfRWr8p1a8LMHYdqTNRAPoOiNnibjEMdBfWDnKDuBuzou1qc2ssaZY3K
6g6M2WaxGF6bLw/ylJ9vVH7b6BJ5EmJFdwHnLwakA5Ktuw/EJor05S82qNfL/mtTOhiH+p8fL15O7qugaldVvWhoFgYGBNr2aPe7Mtp6uDt9A8g7
BYeCJrUIcy2Mf9uNlLMhqHk287OUfXd2Aoq3Z6dvFuB6EnoE+jWmtZw6B9FleQXPx9B2m7DAixfx/VK6fOMHi3K7RhY/EMDQO1EWD1gHjQJqJ/Nj
IPNirNkohDz3I8nzg/PjjFw3P8HPnzJYBoQ74JH9I2X7P2I3evOnnL1fwNj/plSyW9G96zwUZqR9bfl0i94imHyhI8ShJmXx9tW36XE5DFv1ixXY
aEs2VGHYA5A8PvUD5lf0Gz8G64ruVR3o+llHOmBPr21g6Pts2Anq9xt7hBSkTmTqxy5pbbmuFWZrHFEOd4i6EHRpR0O771pYGrKLqmbAauknuKjW
/8fsa2oYYlKSflC7K/4x9MCgxrPuiLNdVxxgTK5b71CUKbGlkrLtNhlM7XmlZD1w2V8xhj57DD3zV87gRi40oJGuFL44/QAFTEAvQBGlzz63pOxn
/fnbixPC0Pc/4spPEN5l/cinwcQf2LEninhqV5SzKoGm04y8Ph3sBt97w+e9WlAaxIP1Ad7D+5pipdYGIgvQU7ciMf4bUErK4lVeUXQw54Qm5j8P
fTlIuPOFyQ7yTxfsKMQEjyNLAx0spUwI/YlG2NLL8fOsbzGvSFkbWfSwLlun6UVG/wXGNF72EZx3QxwNeig5DBH+C1hIRUojnKgtMEAliOTpuCmP
wkBSDuPgIyfgMkL5QEPWjdz+Jfr5/0JSL/xt4fgwYtFLREtjp8WVM3xWwCywY3AbLCi903zYWLD6FQ+Cdp/hHvm6pa8fKIfAoLnGSOEC6WS+6YoN
bmwN+Ia5N9c0uo2/bHH6UmIRThrSPr27vub+HTUI7j/YJC92C95Ig/sw4BmAVKD/gxhJwin3ein772k8QL4INzD1gLQBFUV2xNaCqYijr0e6vJ94
RJh4unM9zz0y2S9xRsX7bia6ovbsi1XlwVSNzjIe8kjsq0/outqJPq5iiaIVGyUL+4opuxpVTYK6iAsfMSL3nWxAQV8USA1R1jl9dlN+yEa4Rzy7
6Q7Kqrx13u3+Bk+DN8ZaUW4lLg9C5rmCUf320+QPUEsDBBQAAAAIAAAAe10kuCtUrA4AAFQoAAA+AAAAc3JjL2NvZmZlZV9kZXRlY3Rvci9leHBl
cmltZW50cy9jb21wYXJlX2F1bGl5YV9yaXZhbF9tYXJnaW4ucHmdWllv3EYSftev6PCJ9FLckRzl0IIBvLLi9UK2Asn2IhgMCA7ZM2rPDMnw0GFB
/32/qu7mObKT6EEi+6iqrq766qAcxznLd0VcSpHFtbqV4urNv0WcpfT33aurN2/fH4k8E/WNFFW8k2KbJ/FWfZGpWOXb9PCVyJefZVJXgeM4Bwer
Mt+JKFo1dVPKKBJqV+RlDXpZXoN8nlUHB3asXINtJe375yrP9P4k325BklZbAmd5k9Wy1PNFXN9s1dLO/YZXPVHLXbFSW2lnPkj6G5cPr1UJgnn5
0HLPml3xIOJKZIUdarZ1GW8fapVUmlyVqOIhyIta7XBgS3SrMhmXUdXsoriq1DrbyazWG3oU7OrfLy8uJ5PBLk/ltgoe8m0epLKGbHb9a37D0T9B
y2kMmae7m1ph807WZY/RMr+PVN4cWBWuVlJGmnZeBnGGzZWqgrjZqoc4KvNlU9VxVJRytVXrm5b/1fnH6/PX0eW//3t+9iE6u3h1fX1+7YuouomP
T354hrimuYurzXF3Ybtdnr3D0JtSpaPDPEOgVLfxFmTKtcpaed5+enURvbp655vHs8v3H65enX2w779dXX64PLu82E9Z3heyVHRDVZBoO4/2iRsZ
beKkpaSRb5OD6vrksFIlCvKTW1iyt/rccjj7bdplk03plvKPRmbJgyV+ffaf89cfL8598fECKrn4/cPbs+vo0/nV9dvL93+Jh7WGCYvorlSQnl3z
AJdwLULhOkAGx+9uxjv49fLsI009Hgj8OEv1WUU3qo53WNZ7iwqZxDejsUou47WKs3bYLOqRWspy2yzjbB1Vcd20C7vhrVyqGzN58HRwcJDKlUi2
cM9oCXdNcdd1cuPCPapTOHyQpXFZxg++XiPHg3m2UinUIAfjLND054UPbCxldYOLPRWrbR7X0ERwMtOEekPHJ544/EWkKqnnKqt9gV+LU31Ox3kX
32uYYZsB5MrDOj/EH/JsQJ4qK+IkMwHk2QF+3+Yf/6UPIDJ5K0sR464BxIIPq7I1IzJR30hZQACcZbWN6yzPvsgyd7tjil9CltXj1TlobWPaQPqa
n/q8f8FzaiUA5HZJUEHe01YvpQToZ+LxiUf0McKWHHi0euIFZX5HKtpizV5IdSHv3Y0spcukfOFuZeYacp74hziiX+bdFzPPx8m1DsMPZSP1aaxQ
0LWrvFPSuWtufU7nmn9eLDyEslIoX3zGtPiiCtfK5ouKYKHWBOn4LMuc1i6sodHdyDQycdCty6a+OeVr9gVAgh4olNmhDFG0fYm320jt4jWNbFVV
z8Fv4ZNRxeWuCsnlWqPRmk4YWKG0Cg4qU7eSdaAoNlY6brgvaMjtMZ6DlDkjnuiIRNvzvP6VarK9y4xVJQVwu5HnZZmXrvM+t7y7FMCc2dGkHrSR
sbu4c9bDfLNgxhtiq7cvPHMORPQKO0xodx/seEmxGrYeLxHJASpPvrWp0RE6aYsB5/HZnxeC7X1DoAYKoXjwgriqHwrpQqPdCiMS0yKL3rQzy4dI
e2DvGHMiGIqjHguE7qxSOqHpVrpsCXO1MDYBU5xaIvComFqhEt+F4nNPQsa6OEmaMk4eOuOw8s2ThfinyJh6AnZaEXQBAfB9V7leT1jSuz3r4wD1
HIYWmTowVvjixMaAzEZXkVazox0u3wTwbTAZgqhjBe5WM1jS+p2MM9pAPJMyj55bilvnpUMNTDgt87rOdy//Ipn56cvFhNRdXlb1s4RGBGaL8XYE
YH0n2PKo7z1Z4NGpmoJiLoazTo94693h0+ACzR2P7vFpxI+AvanojkBrPo99scR+7RAuvXmGXM9IkZ9W9gpdb9FR1G6YqtVKhH23qOZHi4U4HA7N
FiZoZKm8J1ui08IgrI3TKzGWyMVliSzJ7dDQ05zUbj3wbCY13xDpZzw6LeO7Su/BedJ8h/x6hTSnjsps7X5/fDw7/sFjwFwDMjmk9LjC0yh8uEcz
/PhiNGucBDETqRWzWKqM1e9CUIQflWHHur4JxxvnLNaCvQAxqgqPNCmcfB+hO0kpeRWSov8O2URpqn80cYZaQdJ5xAviNtfC/yJmBAj80hvyxTyY
HZ/4Ivj5x5PFMIC2NuBIFAwKGGEjnkEDxnsNAXQfo1k92JvWZzCzFLFw8r6vOWRDmGY48ntYgcqJCpcCc/pY2vFIVy1qdFSQcSQN8yEgcrXpAp09
i0i0ZI1V1kUGyw67dR1JE++XeV4Dl+MigofyaaJE/XyiJUtUUOcU0F2zs81KdRHipkjZtnlBic4pV7C+KNdL+5jAdLlwsANYrigfBUObHfAL7rnL
w4eJAsJEv2iMImRKdEhU5Qgee0qGcfC/gj2iVtDhf+Wco3ZIADjiY6/KfdxD58lpMwsK1t/1RPxKfrFyPmabLL/LbF+BQU+h+OG+AeBJxrVW1yPI
WiZkJb2iBK96vCjzOtfZ5ahOZBeO6xgz+8uz/t0EsIx8eyttgEwVjJayeVMQu0QpoBvNajiUMyAEBM2gqqQOqIAyAlPzouoi9CMu0R1RoXTY63Fm
qKNBGOfeKmQsBc4V1Pe1Q0TiNKrlfe0i0c9TVAWh09Srw58wVxVbVRNZhA26LHoMKNMoXO/Js0k8XJOkpZSSJNfjy7iStBxGK4H5lOebutmFFcOn
9G1obRlMop5S2Fk2yWlwzqHnJVYGRe0Y0Klgu1hOigvg32nluoOt5urInvVio+OvndgmmPpSQF3vnDtUCtthRwctaa0dcevFC33K8UKfQQrejt94
tiaHAfs4LRod6AcSr1DXoPKKkE0lmyIH6BiDwmbLbDq3eOo7VrjXsazg2D9wIIPH6lY6i9YpbDPFknVb5cBrWxXA+oyO1zADPrNHC0iI0ZzEvTSa
H5uXXki35QxUMdpW4/JNRKDcCWiMJBNGWXFt8itZ4de2TzXFbK2PkuF43tdqmw83smeZxvrARN6TLjTgi45LZ0NpLrWMnBnDxbWJcU1wFhfUBXWf
aYX1BKLA0BSMRKYPhYi4XenyEWX/koj31tOPSaKKYQbFG0Yr6WcjuSoA1lA8cZng3FG7iNqlzgJFiDeGuv4PTIMo0EHBzqAC5V8YpAysKQl6pmz3
a7tVgZA7VZOFxZloDPrz1RbAekldNFiEMxWHksJpozSIbEuOj2dUqKz6JlSM1HMcgjzC3Sv9mvyochZBUjSuF3Db2B2UifsbQ9i3XOb3crJ1/3LT
vXV723C33Yv3Z4gUf1NW2ocq4duSDnVoMkPULShmvWBkwAPT1W4B9MSIzFJ2ompY29+p+mZPw57orNR96OhW5SHgUzeIjw61Wx7C1+OKW/+d/a20
bfo9n0VqH9+STem6mvIGX7hdIGOQwIhGX9+AgDesvYcWbgyIzzGY4PY+hqn575LTdWJ41NioNqFTyTUlGCPrhqMB0eIae5iI/lKA4NK2ZiOCXxNh
GeVwzqnj7U2vWkDX8nEO+YjtTyMhDIhz8NNSYGDqHrfW8UKDdD7nVSEdmB5IdxQDQoZ/n4q56ktou9aEPnh3FlMzY5vpLeR3cgidBYf6D8qkvNwg
pw1n1PmrNlFJgSf8fkqQrDsMZrMjn3qaYfAjtwqpLR6+pDqv2CKVDznMaDPhjvdgAOmUeZ9QR7CnyBqSCeqqNiTbG6wbKpiKvtZRWm0P16SlWpFp
6RrH3c0dPuMuLk5mEcqMo58cqrjZqDGniQHJ9yybmBhlc8Z42YQ6QI+XlcucPRSEwdHsOcsa1Qa/xggjqahzwu4yT5tEasvii2ldnjh0iRtc8RYS
wMlhhsz0NHi5ehIooEb2aBHDQjUysh13hnBSOxkxBV198ePQJQl77HZz9ME8d2Iw195GwCO8hivbzstXXeDjlLjTEUp3xmuUaBGll8STHXXBrLuW
Y30jywir9cNkj25a9neYLq1rWyER0AA+2ZJYe/zNdjRtCPe4eCMI23Obzpsyb0BLn9qkYPShAAVfvBVLWd9JqT8Im8SIZHb63U5sNOHAfXQZh9vW
e9vhTIaJy1r3NfUSk2fs/RZpKpOqWcJkOYY4Rz+1HTXmzbm4rAB2yH5RqVfqPjI9f2q6bU7Frb5EHw/c+SKBTQONVc1tudsFTfK3raenXllSDU2B
wKX3DYj2uG7AXZQT+ihhnn884eejWTd+Yp57d0JRcxhPetY27Hnb5a1TTFzVLbTiEyZStAY7Wdi/kulnsxet78yLxfz4dKG/bIWjg4ftk2d1uS+P
1EK1ah8s6U5tND1fcWYSPfKntOB49URZUvTYfXGjMWd6fN1qHH2ZWSMJeIz3WQDrMV70bYBbi+tat1xjq/6n5z4DjnjD61jV5jsOf3CZpO+00AcP
buZqe55ozhreuAm3r9gUSCq4Od+FTh6gQnVSmJ3qqqzX1+otGTTndN0PGih5VIaCOpL3MmlqZmVi5P4Kzk53PPbVu3Ql+8tdrXvE/VHW9tUcrXdF
DqXVbSg8Ppr2Dt0utjDojndQjP32jdufjphB/Sm5fiNxHKn/hHj7coA/L98eAfdkC32TsOshUfvsty3QYYeBWxfstmSJSGIb2uZcnX96e/6/6Prt
+zcX59H1+flrZ9gS3cGqXG5evs8z8z2Z/zOIupv2v4SCV+W6oYT5N55xU1klpSqIbxhFaZ5EkddCsv2y4Dq9Vh4lzage6E9b6tNL3tRFgzy8B8LM
IojTFMm25ooc5/CQP2A8YQ/VU6Fp2iLSki56n5z3bcdunbVSV0x/kAidmfPVDTrRT25ybKtCt/tPGK+j0fsnDKKE7RSdDEH+QyQtoJCDS/5yo5vQ
NBX0VOQzgYBbePzUKsq828wbooU80PZb+U2r0jQig90mVaWrXyrWD4q/ewWcyDc9dfX+x8TtUfG1tLabC2gpH3TLtpHzZwxwoQvZaIaQy4FihvBq
GnogAbVyPzFFfVu5j89aseHma/zoOaVh/iyq7Kll+q4+pfQ8AjxPqvUsQ8O8LzpT5i8DfV0+U/eLDtJHSG5fn3z+epfV4bGHIv4AcTGKyAuiiNqP
ThSR80aRo51He/LB/wFQSwMEFAAAAAgAAAB7XWH2PZbFFgAAkEoAAEAAAABzcmMvY29mZmVlX2RldGVjdG9yL2V4cGVyaW1lbnRzL3J1bl9hdWxp
eWFfb2ZmaWNpYWxfZnJlcXVlbmN5LnB51Txrc+M2kt/1K3DMF2qWpm3Z6zjOaau8E+dRN5mZ8niytetSsWgSkriWSC1Beuz4/N+3HwAIPiQ5uUtd
3VQyNslGo9HvbgDjed5lGm8q8Q+ZF2khyjqP7mIlV1kuo4fTcPMkqkLEYhNnpUwD8RCvsjSusiI/KPLVk1BJKWUeep43Gs3LYi2iaF5XdSmjSGTr
TVFWIs7zoqIhajQy78rFJi6VNM//VEVuflfLuspWjC0pgAD9Hn/nt5u4Wq6yO/PhIzzyh0quN/NsJc2XG4k/4/LpO6A+qYpSI6ieNlIZoE/wcyXf
x2upNnEiLY0AnizNQ72qynj1VGWJYhTOC4Po7x/efeh9DJP5wgAsZBXBYx8GOBoboLs6W6XRU7EqInytZNWHXxepXKkQgUIlF2uZV3Y1/Ej8vilj
EGM5Mrycz6WMUlkRK8I4B3QqU2Fcr7KnOCqLu1pVcbQp5XyVLZYWZaSW8eTPZ1uw8OB5Kf9Vyzx5iog2M/Ty+udPgfjefHSJ+xnhduJcx+p+YjD5
IwF/fr789F+TCLAG+teP1x9uPrz98C4Qb4v1GrDCmB/KLP2F9bQoA/FjHJf4ekIzBmJTFlVBGhDFVbQA4GA03klImYHaAznlIssNPdc//XL5jinh
X99+eH9zffn2xjw3lF3j8J9p9CsZIB83sswQUoUgDjAVGWlaADJLMiBnXqxSawF2Lm2fsg23fxa0+44eWJG2RRBt6rtVppYRGlogolIqQCbxJ6io
gjeqXgOvsl8lSRDffCkzIAmNHFk9+vnDd1fvxFR4qMCTM3UAOhxuKm/0t6uffvjx5lP0+Ro/z71lVW3UxeHhIquW9V2YFOtDxwwOYwXWoQ5LuZJg
J+owLb7kqyJODx/Ow9Pw6PCZJnpp8H768RIUGWc+SePj9PxEnhwnsUy+OZt/c3J0JO9Oj85O56fzs8n5+SQ5PjlJvk4mZ6fp2Uk6l2fH8dfJ8Tdf
n598HZ96o8/vQNzv/n7z09tP0S9X159++vAeEePMk+Mjb/Tp7fXV1XvUkE/w3n/DpuAosNWg8Qj+svqC0M8W7KKn53bYRUfRXmDKH6+++/zuCjGQ
rDy5KZKl8i7E+VEgPPCbGQhUwvPkz/CcrRfqV3g4O8Wvd3GVLOHp+AwevhTlvSxx5Ck8KSlT+PUoYKzFpsrWIN8S3mH4WP/NA6BVeYQw4dHRMSKQ
6EJA2ZL4Sb8+wjmTQkUrHHhT1lLjW6qHaMlAxwiDz4qeT/XTAz0R0SD+XK1AxekVTqWSeMVPE40Q9FptQMmzB+nOvS5UnCX05gQewcttiJSjkL5m
j/WGwYPRy2g0SuUclDlZyrReSbCj0o/L9YVQVTkWB38RaZZUFzRfKSHi5eL5zRsjAUQHuh+VaOnI7heRzSHqrcV02uiAACcuhRmjZ6zYaffnQ5d1
O+DeZ0yExg+xVoCXcrSPPxOdcQbzgWus5VVZAv659zm/z8FqBM30DH+/eGCgCJyswLzEB+1G9Fz+wPzjZgKkH4McxQAfsoh5ICDgTd8XObgKVgml
nx5keVcoOUU9cFA4SwFWedc//NVrf3QYrmoQtD8OmylhMjuPnWLcGg/ONF4rNBEvR11AKikA3+LzDFV02X6dLCGFAVHRR40TIPRvL1tIb0y4Rz6H
yGknMjHxb94wgW2iIR+ziBsHsBXxroizexolt2PdHsYZJ9A3hf+3ITdoiLPKiCzKMfFCbY/Qdfv0ctzlqRapyBQpOGrQFjpDwqLh23i01hDYqKWz
nHPpdEvrLTjHCBPNgAZMPbJL8HPkJkmHh7X2P1wDL0oaTarMCPYqc5uWNhV69vayvoKcbwXOTtxBJIYwKopcVEspFDBWbIrV0wJelLGqZCliRZ+W
GcbsDPym+PHy8hrpDls4QXXQRDDlJnaE+GLcAwkbJwfQp63v4LYyXLlYx49+ljNbOXMN+VuIX8bjQJxMBgXVz4R9nNQVDbEjaIxVC4t5hSk/uZdA
EzPlH+NRz2E9mHSRqNwv2Fe5pAZpn3P/Q+bq2bbkvIy1AnUgm5KQBKv4ATK/rJyy9emngCaaMlcjUIfVXZzcKwayjx1+zTMoHCIJqxti1lfiI2SC
snyQwqYJUAKmIIJis8nyBfwCwVuAwUOcAp0s6w36EZRWAS71KRQ3S9lBCdkpBBqBc9YwmJVYkdbLlPReQJhO7jcFoET9b8pU+LoqvoSdEKDUyI3c
nShnYr/MFVayMDssRBqnckEFJ4Xkxg+BjrgFWhTBUhRMD2Uw6M1ArtiNytc8iYnLV48bXtxnp858HsDzEgAra+Dv85b5MaBrCnUhZ70jktZOjXfk
Ch7IlXOTVFDiLdJCsjteoxWSTEzVQbWwzuwFoD4A3JhYEF9tgemn8gGks8Fgwly1wdthcpNneZ739uPnAxXP5begPFkFM4F+Ca7p3emZQFQ74EM2
f9L9CxtHvkBBUdQVJ1uglNTAwCm2yJxZSGX6dEuN5S4mBBMoVg8SvBvnUkUOM4GLnlKng2KU8n3EF2JxB0p7CGm1iy4yY0Ic4Y0BZZxGlXysfAjB
RQpET726mh+ce3qSPMZ8F6ZA3vvg6izxgQDveT/1dKsAkHEMxFHcSsB0iBMZtEv0dpBDtpNHnZcCZHH3T9BN0LFcfokiv5OwjsfdIaHmm064JseU
Y+mc6kJgKk5JAPyOsob11WsJfk/6hge3GmA2Hr/0sGt32mnj+MlKQU26jjecYdphKHru7YRQSaTFOgTCwc/mC5Rglkg1vZ11nBqDr+O8BulgMeSb
pP2Wa6PZQKqjboEbM6DMENqkqSyq8ClerwIttyYh/j6GLGwckoNltKYhh8GUMVNePAvJlUbENIakmN+VJeTgsFBSWRSrbh5BRb5WvrNQ9svTBriP
3kxhVkYA4b18AkSY4xhCzSs0QIgi/gAzIZXEAIIIbgEaUusKTK+AUgephjfmBZJs8WqiLT67ahA9ctqK1fdBy0CzvjnD/8ECFjJHnSrKKQP9YJ4h
WLuCPdb666hJls9liXUzSc9lGbiQTV2R9VABpeseImd8ezSD/xoh6MZYVwAsqjjRxsuI0Ku6fNITEd8DM63WgrFrs753+f0Eq3HM6zyNHFw/IkRJ
oPRCDKMSph5jfmq+MAXD3x7DTM3R4cLqQ3zBcz7ijJqYZuCeyPaRPTH4bIj+qB91mlViHmcrmV6wRk2fkUyIbEzT9Jl/mkhmym2POnlJsQLH0fRH
PPSe1JhANiwluFIg0b7I1vGCXE3jW/SrWdBSUw9piHSUiYguLAvhJc6hqY9i02TgXzooKAVj9AAJrkVRH4UsPNAVATjySD7KpK6aby/dSMm90N8R
L28wN5KPGHtswKS8CNNjHacFIherQkESSBYLqVXJoTUVVvlFiWnGAWTQmz84Wupk/w8Nlqqoy+T3B8vbWTdY+mxxTodv/H8bOzUX/3+GzjDebMC8
/H7oZMHp0MkPW0JnEzsDrFma6NmKqW5w5ADWCns8tB8J4zQF48BmRuVbBGB+9MLg1skfiGtBwLf3HNyA4KGohn763jTwGqq0H3bDwYND9e39bDwz
bt7MBWNWMveJSkrzjw2aHI0CJiz1R+BWniqUrO+F5FDDVFbgjUM2CG+fP6cGFkQJQAkT6ySUQzd4JVBzcN8408u3hrrps/7FliY5RJBo+QX75Biw
jyfn/OGxE9MnFNPfGPjfENZPfktYBwZErDfEEPpdK4f/iHG9BTknLWJIrWf4w4FMCtzrhPf9rSe/GRdYPgTi1NZsflf2DXFm8rYBgZwbnCFUfBt5
ezC5mKEWVDXYuZ+Lw0MxIVXMUdUsO7t4ulM33Aj0mvZGe9aOdugBTzUH1oN23BWPhyopSnmYFHI+x1aRiO/U1GtRMvee7aoP7KJDACQhP8LfaEGQ
SmkxWDzPDskHmuLhcd0ZGQ9xb/rMbOtydWzUl3Sqv9XsY/TOHqe0EzA5sLH8wBtTHw4GNMzDdhYoCAZxH7+MMcZRxKWAi5tjbY+KLRz/2SOPBs6f
FYgzigjdOrx79jCSYeZjYtkLcgjzBIvMaZs4sZBgeoHQDiJHY8aFCIXKZQHbAWBALbxrSX2p1J1et3Yx+Wi6y+yxnalNE9mZn1uKTrn0ajs3FlwS
OcaG+clEi47B61S6a5DM/gbXeLv5uO5CA+9n2NuGT00aZr09G5lOxDWzvhLvC5tJUQpKuV1eOG05VclNKK4eZZnglNhBSQpMBaqmlaKRXU2uKENk
xqqnHICrLBGbQmXk8bMcYiQwWZn2M2RjsGjdeT2YCPR23IYjE9Kphz4S4WPrDwHBZnta299MGzd4SOO14Gm72YYMWGGhKGacngfi7BQUOkXfqyNF
Dcw8d8bdXmDguTjB+HN2cXqO9e2xThWwvYU5WLZewPyPZrM0ytJHeGZ8HPL82yMYPxtjmkYZWvvj7VF4BAXc7TH8ACArd+8OXKEcgMedyeZ/HNp5
Q1PRAmAw77c3WJVcR61vYbIqIDTpVJBT/sgELGdvoaXmtvZDeF3/7XT7numntm2Z9EebOWPUuoofQlWvcfcjTu6/xGWqJbooyRJJ0mSOtwfHs6E0
JURIQzqNgomwPds2RLsUBDGlbAsiAfOqorzIUX0YbPdab2g/ZbGU5QHVMTX7LVNSQbBLCT2UJLpJyoSLO8hpkuWOmra78+9xcsW1aERFndlHh1gR
f9EnO/CgVftroxAJbRhQ5nFqa1snK+kgtV8iNmM0Pgcb5XUR7fXBzHRaoElLe4nmLMTaZIU5tWcdSIQ8irQ29MhV9d0me5SrCOWQyUYwDYlNINDe
dNvStxbbwe463Z4EsIUsnnSBYreZORCtjXp4wypjNnqaWAU/oSJzojRHFjqmMQRJH7yZ+JN2RZgQgsuMKwjS3RBIbQlzCgsIQr+JDSLK/+F5KLs7
Fv851bPDL01lps+NzLpjOgRiXeY7x0HGxuq64+L8ifbuaMA9UfTQqoV65yzGpirak2JCLoHScDOJXfsSZhrBlXvb+mjVRtzIyozONen0wcdOgyxN
46UCg67QudnmA5fgHWWAn+aQiGmktPsajBXTvWbTSh+menU3Q5rdoikeQulTBpqAi73gRuSWvplhDbw1imDbh76lnkWOx8UMclYwSwIwtgNMfN6i
hB1QvcOHLFCbVaZxI2e8AehBsx2bEEPWu2c+x3vw3hhPaDbKGtno7pCHv+t9dm9v8UPnaZruPeZfVjZiXSs86xqX0qRpgd2Ows3QbAOe6YePn3nz
tJhXXxBUb+uptuLaVd0OrGimNZqO+JVre6gI89Xd7UTTaTZP3OThwS3GvgnEMo7LxlZ4hPhvDsFTwUd+ysXdbpBO//J/8UiTRoeoeBWQCqTA5HWs
MwA8ocBfxuIvJoVM6jQO+W1EicG+xOemESB5DwXSE/EDhPv4DpzO28/fXer5NU1/TAcVz79lkN1TVccybEBAhY0hWm84HXDA3Z3L5yaWOg6kdXSR
jIp8jHnl5riuC2o16U0RS8c1257I/Bq4k5t9aKPhF51dbEDhbIbD54F9czczQFl7pNY+y93ZPsdsZbGpbWZOKoFVi1YL7K76jvK4C3YkZL1lQ7Nx
Mr+lsc3IX1zrGD5qBh/bFmki81Ct6e7yYxc7IoSNDmMElY+gUKjZdGAI1BXsVDc6nPK82VRpzx4NuCVUun6QbY8LhBPCeEurs/aB03DwseVpXrl0
46+Hlw2Z1CtW3Zr39YtuDWutmRI4O9HeY9y0mRbtOLbe7Ci5X0cDS6HznxnoIodj3rjAvS5cQ/vIe9PFGsTueirndCYbzHTuoUldPPOj8dUwOfoc
x5F1gjAC2LYYnQDaBW1Cti738WxzD97Nadjc2JeWxQK0QfXgzQcHltO2HuSWxM4uVFt/uL4H7fL5QemTa6SAUXHvbMVkc72EkD62NvHhm5NcarBd
+SOmOzaB3duHwmIXrcCGBd2C+lbUABtju1/Hm56h0BFWWq2hGsMuCcZ9oRnkvrKM7i93iMbPOVSq8GKeYTbKxMQlPAO96lWEuodgnbsLmp2NfVpx
dIh26JNkda2MXwPvTeo1bp9xbMu7rSSIdw3knswW+d7u9+OfDo692XgD+opU/BVtzivjbbUh8akAkVUKte7QqfEaJji81rkwUjV6VWeeHeWBCbmD
nflVgcdkO615t5IhgEg7LIbe5auaIdprDQ5pOaz2LNu9hXNIR5/lcdTOWtFOb5DN++bG1xSeX1ytbBsyaMVdUejjPAN23f6+Xw9+ym0f2pE5XTzi
cPStOS4KpgyqYHytCxxnpZhnpepsXBAJbQqwD2F2klOUCW7n+7jCwGE7JIJkNYEjvXFnLXoOY2SMlJbPB50gxdD710Dv+sVmBX0s2zgz0O3IFkiz
3lPjFEFf0QLTMcL0+sbO9yxDPIaMV7l8lwXd89g4aWT2Xuw2UYs52zaL8I+RJjYoen00Fzs1T3qn/5vhlpOmMzZwEHv39gk3u9rcMhGe/XuH9u4F
N9/aq34RJurBC1pMcppj3OvToAD2/OK49/59C5dTR7uE0OqybUqsQEwCSzcDLwXue4sjKIEtzpfD8yNmgKID4SnQM1/VatnxHrpQ5tN9zSqweWpw
wcghp26oaxGrA2Tz3FziYNey/wxOaxK0vEir/dSlkj6YaDdqjaHTVGwV5lDJgO3mRc7SdaYYMnBsZ5o2rHuWJhBDvU1Tw9qO5uvNXR9Itzt9WJLo
o8y2SO5bduuKpiVQO7R4KPBn9t5ZSEufZ1VOOW9zetW80s7dCGCwttpNCHvR4QykL1/jTN3Rr10yWGbQzfIdox1A42R7xi2AzWpvc2GZwV7kT+KY
Nt8es3W9juxdy34bPRhc54BVXXCcNNPgnQlsQKDQ+o0DV7AI5ar/hcvFl44xsI+K09Re+fA9yPH4PhY6BrBuzc+Bw1l6y1XPPu0eYQuobYVXbaiz
MXbvnpiYMe4zhD3EtBNW2t6fnIXes9HVI/8AclcFJEH625s3xjic0plurfzevKjrGh1stwNinKGVGDE56QL6htZQV2SzV5Rfpr24jNmHAvFpneCN
DMZLRzY792EGTk90IjjRtvO4h3F2rACv2m7av5q/du7tYEenFZjR29EUCqXfbNfgn6Y2oZ15e9cKNH/rvwTQPkZwuuNOrpPsmktH9kQInnT2ezqO
9dGUSiO6n6Z+nTZugO9Zd7wA3yNsgPj69SwQ+uq188lcxp511b6FEHQBWzSulQnsTk5Zy5omhNczl4aZjq7IqsRLR9PehX7fsgT/8Ymg4VBIR03N
zoSufJ9bfV9rzI27bBmETusAsld8XnDl6fRXNYl4uIF/C9pdHOeUd3un2DrJ7cbo9od37g4jiBs2OMAEmgGdbRouTWk7Zk3nVtq3yOjfRMHOlPn3
UcLLclGjOX6kLz7k5kmZ0YW5aRSlRRJFY2ck+fRYD/G9gwOn8wZSp9MvvJWje5Cpk/ltQWGq0t85nJss7uid4OxIkmVBp4mdLZ99y8QtFTSPOdT0
YIhH3s4B2GU+sB3XV1NXLu5+x6i4rpYFms+BiRQeX20AIXpcW1TARk2xPjSgcdEPOirgO53vBXZ6NdLIIG01ARkGj6Y3x2DonW56DaZtek7J7tnZ
jNQNcRIP7UdauTb+gu5/YOpvtxmZAr5jGrZ6wPTGNoIduhrgnntrbwzQndWwu1nQ6qMzSOuV04m0DBreQ/jjO+2GX1t65vuYpgOBw66h7qWZxO/c
JBEdDnTCn4Ue76FD6z2Vn5RTpfV6o3yaFs8zYyN2OhnjzU/si9BuWRTRdfgoQg8YRfq4KLvD0b8BUEsBAhQAFAAAAAgAAAB7XcTs8ae2BAAAhAoA
ACoAAAAAAAAAAAAAAIABAAAAAHNyYy9jb2ZmZWVfZGV0ZWN0b3IvYXVsaXlhX3JpdmFsX21hcmdpbi5weVBLAQIUABQAAAAIAAAAe12q8J764ggA
AHAWAAA8AAAAAAAAAAAAAACAAf4EAABzcmMvY29mZmVlX2RldGVjdG9yL2V4cGVyaW1lbnRzL2F1ZGl0X2F1bGl5YV9yaXZhbF9tYXJnaW4ucHlQ
SwECFAAUAAAACAAAAHtdJLgrVKwOAABUKAAAPgAAAAAAAAAAAAAAgAE6DgAAc3JjL2NvZmZlZV9kZXRlY3Rvci9leHBlcmltZW50cy9jb21wYXJl
X2F1bGl5YV9yaXZhbF9tYXJnaW4ucHlQSwECFAAUAAAACAAAAHtdYfY9lsUWAACQSgAAQAAAAAAAAAAAAAAAgAFCHQAAc3JjL2NvZmZlZV9kZXRl
Y3Rvci9leHBlcmltZW50cy9ydW5fYXVsaXlhX29mZmljaWFsX2ZyZXF1ZW5jeS5weVBLBQYAAAAABAAEAJwBAABlNAAAAAA=
"""
payload = base64.b64decode(''.join(OVERLAY_B64.split()), validate=True)
if hashlib.sha256(payload).hexdigest() != OVERLAY_SHA:
    raise ValueError('Paket kode kandidat rusak atau berbeda versi.')
extract_sources(BASE, WORK)
extract_sources(io.BytesIO(payload), WORK)

ENV = {**os.environ, 'PYTHONPATH': str(WORK/'src'), 'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '2'}
def run(module, *args):
    subprocess.run([sys.executable, '-u', '-m', module, *map(str, args)], cwd=WORK, env=ENV, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check',
                'ultralytics==8.4.210'], check=True)

In [ ]:
metadata = fetch(
    'https://zenodo.org/api/records/24346584/files/splits_metadata_results_scripts.zip/content',
    WORK/'zenodo_metadata.zip', '798bb53b0cfad35dfcfbcc463fbcb79ff4af61f9807803d6bcc65d37f7c8feaa',
)
weights = fetch(
    'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt',
    WORK/'yolo26s-seg.pt', '3da1d83e31caec96f9300eb4064f4f62882c133c7c264d63dfe61a7c197837a4',
)
DATA = WORK/'fold_a'
if not DATA.exists():
    run('coffee_detector.experiments.prepare_auliya_official_fold',
        '--metadata', metadata, '--pilot-bundle', PILOT, '--output', DATA)
RUNNER = 'coffee_detector.experiments.run_auliya_official_frequency'
args = ('--development', DATA, '--weights', weights, '--arm', ARM)
ref = json.loads((RGB/'validation_result.json').read_text(encoding='utf-8'))
if (ref['checkpoint_sha256'] != digest(RGB/'weights/best.pt')
    or ref.get('evaluation_split') != 'val' or ref.get('test_images_accessed') is not False):
    raise RuntimeError('Kontrak hasil atau checkpoint RGB tidak cocok.')
print('Runtime RGB:', ref['run_contract']['torch'], '|', ref['run_contract']['gpu'])
run(RUNNER, *args, '--device', 'cpu')  # sintetik; tidak training atau akses test
PREFLIGHT_PASSED = True

In [ ]:
AUTHORIZE_TRAINING = False  # Ubah ke True hanya setelah preflight lulus dan GPU T4 siap.

if not AUTHORIZE_TRAINING:
    print('Tidak ada training. Preflight dapat dijalankan di CPU.')
else:
    assert PREFLIGHT_PASSED, 'Jalankan preflight terlebih dahulu.'
    runtime_code = ("import json, torch; assert torch.cuda.is_available(), 'GPU Colab tidak terbaca'; "
                    "print(json.dumps({'torch': str(torch.__version__), 'gpu': torch.cuda.get_device_name(0)}))")
    runtime = json.loads(subprocess.check_output([sys.executable, '-c', runtime_code], cwd=WORK, env=ENV, text=True))
    expected = {k: ref['run_contract'][k] for k in ('torch', 'gpu')}
    if runtime != expected:
        raise RuntimeError(f'Runtime belum cocok dengan kontrol RGB: sekarang {runtime}; perlu {expected}.')
    arm_dir = OUT/ARM
    arm_dir.mkdir(parents=True, exist_ok=True)
    progress = arm_dir/'progress.json'
    log = arm_dir/'training.log'
    command = [sys.executable, '-u', '-m', RUNNER, *map(str, args), '--output', str(OUT),
               '--rgb-reference', str(RGB), '--device', '0', '--authorize-training']
    print('START/RESUME:', ARM, '| LOG:', log, flush=True)
    last_epoch = None
    with log.open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=stream, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                if progress.is_file():
                    state = json.loads(progress.read_text(encoding='utf-8'))
                    if last_epoch != state['epoch']:
                        last_epoch = state['epoch']
                        print(f"{ARM}: {last_epoch}/{state['maximum_epochs']} epoch tersimpan", flush=True)
                time.sleep(15)
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if process.returncode:
        print('\n'.join(log.read_text(encoding='utf-8', errors='replace').splitlines()[-55:]))
        raise RuntimeError(f'Training gagal: {process.returncode}; log={log}')
    print('RESULT:', arm_dir/'validation_result.json')

In [ ]:
# Dapat dijalankan sendiri setelah hasil training termuat. Tidak membuka test.
candidate = OUT/ARM
if not (candidate/'validation_result.json').is_file():
    print('Perbandingan menunggu hasil training. Tidak ada evaluasi dijalankan.')
else:
    summary = OUT/'paired_classification.json'
    run('coffee_detector.experiments.compare_auliya_rival_margin', '--development', DATA,
        '--rgb', RGB, '--candidate', candidate, '--arm', ARM, '--device', '0', '--output', summary)
    print('SUMMARY:', summary)